# OdiaBench — Model Evaluation Notebook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tripathysagar/odia_eval/blob/main/notebooks/eval_odiabench_working.ipynb)

Evaluates any 4-bit quantised model against the 5 translated OdiaBench reasoning benchmarks.

**Supported hardware (auto-detected):**
- Kaggle 2×T4 (32 GB total)
- Colab Pro / Colab Pro+ (A100 40 GB or 80 GB)
- Kaggle single T4 or L4 (16–24 GB)

**Default model:** `google/gemma-4-E2B-it` (Gemma 4 2B IT, 4-bit via BitsAndBytes)  
Swap `MODEL_ID` in the config cell to evaluate any model.

> Needs a GPU to finish in reasonable time. For a **pure-CPU, no-model** walkthrough of the `odia_eval` library API, open [`capability.ipynb`](./capability.ipynb) instead.

## 1 — Install dependencies

In [1]:
%%capture
# Install odia_eval (latest) + model deps (transformers, bnb, accelerate) + datasets
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-qqq", "--no-cache-dir",
     "odia_eval[model] @ git+https://github.com/tripathysagar/odia_eval.git",
     "datasets", "tqdm"],
    check=False,
)
!pip install git+https://github.com/huggingface/transformers.git

## 2 — Verify `odia_eval` (benchmark data is pulled from HuggingFace Hub at run time)

In [2]:
from pathlib import Path

# Detect runtime root (Kaggle vs Colab vs local) — only used as the
# destination for the saved results JSONL.  No dataset is cloned: every
# split is pulled on-demand from HuggingFace Hub in the run cell below.
if Path("/kaggle/working").exists():
    WORKDIR = Path("/kaggle/working")
elif Path("/content").exists():
    WORKDIR = Path("/content")
else:
    WORKDIR = Path(".").resolve()

# odia_eval was pip-installed in the previous cell.  We use its prompt
# builders + scorers directly instead of run_eval/run_all because the
# library's HF Hub loader hardcodes one split per benchmark, which would
# stop us from evaluating arc/test or arc/validation independently.
#
# _strip_prompt and _aggregate_majority_vote are private (leading
# underscore) in 0.4.0 — still importable, just not advertised.  We pull
# them in so hub_run_eval stays at feature-parity with the library's
# run_eval (prompt stripping + n_votes self-consistency).
import odia_eval
from odia_eval import (
    BENCHMARKS, build_prompt, score, EvalRow,
    save_results, load_results,
)
from odia_eval import _strip_prompt, _aggregate_majority_vote
print(f"odia_eval {odia_eval.__version__} installed. Benchmarks: {BENCHMARKS}")
print(f"Workdir: {WORKDIR}")
print("Data source: HuggingFace Hub (tripathysagar/odia-*)")

odia_eval 0.2.0 installed. Benchmarks: ['arc', 'gsm8k', 'hellaswag', 'truthfulqa', 'winogrande']
Workdir: /content
Data source: HuggingFace Hub (tripathysagar/odia-*)


## 3 — Hardware detection + batch-size config

In [3]:
import torch

def detect_hardware() -> dict:
    """Return GPU info and a single uniform batch size for a ~2B 4-bit model.

    One batch size is used across every benchmark (sized for the GSM8K
    long-generation case so MCQs are merely under-utilised, never OOM).
    Override the returned value by editing ``BATCH_SIZE`` below.
    """
    if not torch.cuda.is_available():
        print("WARNING: no GPU detected — eval will be very slow on CPU")
        return dict(n_gpus=0, gpu_name="CPU", vram_gb=0, batch_size=2)

    n_gpus      = torch.cuda.device_count()
    gpu_name    = torch.cuda.get_device_name(0)
    vram_per_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    total_vram  = n_gpus * vram_per_gb

    # Conservative single-knob heuristic for Gemma 4 2B IT NF4
    # (model weights ~1.5 GB, GSM8K max_new_tokens=512 dominates KV cache).
    if total_vram >= 70:         # A100 80 GB / H100
        batch_size = 256
    elif total_vram >= 35:       # A100 40 GB  OR  2×T4 (32 GB)
        batch_size = 128
    elif total_vram >= 20:       # L4 (24 GB) / A10G (24 GB)
        batch_size = 64
    #elif total_vram >= 14:       # single T4 (16 GB)
    #    batch_size = 64
    else:                        # <16 GB fallback
        batch_size = 16

    return dict(
        n_gpus=n_gpus,
        gpu_name=gpu_name,
        vram_per_gb=round(vram_per_gb, 1),
        total_vram_gb=round(total_vram, 1),
        batch_size=batch_size,
    )

HW = detect_hardware()
# Single uniform batch size, applied to every benchmark.
# Common values: 64 (T4/L4), 128 (A100-40G / 2×T4), 256 (A100-80G/H100).
BATCH_SIZE = HW["batch_size"]

print(f"GPU : {HW['gpu_name']}  ×{HW['n_gpus']}")
print(f"VRAM: {HW.get('total_vram_gb', 0):.1f} GB  "
      f"(per-GPU: {HW.get('vram_per_gb', 0):.1f} GB)")
print(f"Batch size (uniform across all benchmarks): {BATCH_SIZE}")

GPU : NVIDIA L4  ×1
VRAM: 22.0 GB  (per-GPU: 22.0 GB)
Batch size (uniform across all benchmarks): 64


## 4 — Config  *(edit here)*

In [4]:
# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID   = "google/gemma-4-E2B-it"    # Gemma 4 2B IT (official HF repo)
# MODEL_ID = "Qwen/Qwen3-0.6B-Instruct" # swap to evaluate the student model
# MODEL_ID = "sarvam-ai/sarvam-1"        # Indic-specialist baseline

LOAD_IN_4BIT = True   # NF4 quantisation via BitsAndBytes

# ── torch.compile (inference speedup) ─────────────────────────────────────
# Compiles model.forward via TorchDynamo + Inductor → CUDA graphs.  On
# Ampere+ GPUs (A100/L4/H100) this is typically a 1.3–2x throughput win
# on long-generation tasks like GSM8K; on Turing (T4) the gain is smaller
# (~1.1x) and occasionally regresses on older torch/bnb combos, so we
# wrap the call in try/except below and fall back to eager on failure.
#
# Requires: torch >= 2.4 (ideally 2.5+).  First forward pass is slow
# (~30–120 s graph capture); every subsequent call is fast.  Pairs
# especially well with n_votes > 1 (more forwards = more amortisation).
USE_TORCH_COMPILE   = True            # opt-out: set to False to skip
TORCH_COMPILE_MODE  = "reduce-overhead"  # "default" | "reduce-overhead" | "max-autotune"
# ── Attention backend ─────────────────────────────────────────────────────
# "auto" tries flash_attention_2 → sdpa → default (silent fallback).
# Pass an explicit string to force one backend.
ATTN_IMPL = "auto"

# ── Static KV-cache + fullgraph compile (opt-in) ─────────────────────────
# Requires USE_TORCH_COMPILE=True and transformers≥4.40.  Often fails with
# BnB 4-bit + variable batch shapes — the loader skips with a warning.
USE_STATIC_CACHE = False

# ── Length-bucketed batching ──────────────────────────────────────────────
SORT_BY_LENGTH = False

# ── Reasoning prompts ─────────────────────────────────────────────────────
# If True, wrap prompts with <think>+\boxed{...} instructions.
# Also disables the \n\n early-stop heuristic in MCQ generation (paragraphs
# in the thinking block must not truncate output).
REASONING = False

# ── Self-consistency (test-time compute) ─────────────────────────────────
# N_VOTES > 1 samples N completions per prompt and majority-votes the
# extracted answer (Wang et al., 2022).  Requires generate_fn to be
# STOCHASTIC (do_sample=True) — greedy decoding produces identical votes.
# The actual batch passed to generate_fn becomes batch_size * N_VOTES, so
# halve BATCH_SIZE if you bump N_VOTES on a memory-tight GPU.
N_VOTES = 1

# Per-benchmark early-stop flags (recorded in .meta.json; gsm8k never uses box stop).
_EARLY_STOP_ON_BOX = {
    "gsm8k":      False,
    "arc":        True,
    "truthfulqa": True,
    "winogrande": True,
    "hellaswag":  True,
}


# ── Eval ──────────────────────────────────────────────────────────────────
N_SAMPLES  = 64      # rows per benchmark (None = full split)
SEED       = 42       # random seed — same seed → same rows across models

# Benchmarks to run — all 5 splits translated by the odia_trans pipeline
# (see ../../docs/translation-todo.md and tripathysagar/odia-* on HF Hub).
# Drop "hellaswag" for a quick 4-benchmark smoke test (it's the largest split).
RUN_BENCHMARKS = ["arc", "gsm8k", "hellaswag", "truthfulqa", "winogrande"]

# Every scorable non-train split published on HuggingFace Hub by the
# odia_trans pipeline.  Each entry is (split_label, hf_repo, hf_split)
# — the split_label is what shows up in the results report; hf_split is
# the actual split name on the Hub.  Each (benchmark, split) is
# evaluated independently so you get a separate accuracy + Wilson CI.
#
# Train splits are intentionally omitted — they're for fine-tuning.
#
# Splits deliberately EXCLUDED (upstream hides their labels, so the
# translated mirror has `label=""` / `answer_idx=""` for every row and
# the scorers silently degrade to "always A" / "matches empty"):
#   * hellaswag/test     (10,003 rows, all label hidden upstream)
#   * winogrande/test    ( 1,767 rows, all answer_idx hidden upstream)
# ARC test labels ARE public, so arc/test is scorable.
EVAL_SPLITS = {
    "arc":        [("validation",    "tripathysagar/odia-arc",          "validation"),
                   ("test",          "tripathysagar/odia-arc",          "test")],
    "gsm8k":      [("test",          "tripathysagar/odia-gsm8k",        "test")],
    "hellaswag":  [("validation",    "tripathysagar/odia-hellaswag",    "validation")],
    "truthfulqa": [("mc_validation", "tripathysagar/odia-truthfulqa-mc","validation")],
    "winogrande": [("validation",    "tripathysagar/odia-winogrande",   "validation")],
}

EVAL_COMBOS = [(b, s, repo, hf_split)
               for b in RUN_BENCHMARKS
               for s, repo, hf_split in EVAL_SPLITS[b]]

# Output file
RESULTS_JSONL = WORKDIR / f"odiabench_results_{MODEL_ID.replace('/', '_')}.jsonl"

print(f"Model        : {MODEL_ID}")
print(f"Benchmarks   : {RUN_BENCHMARKS}")
print(f"Eval combos  : {len(EVAL_COMBOS)} ({', '.join(f'{b}/{s}' for b, s, *_ in EVAL_COMBOS)})")
print(f"Samples/split: {N_SAMPLES}")
print(f"Seed         : {SEED}")
print(f"torch.compile: {USE_TORCH_COMPILE} (mode={TORCH_COMPILE_MODE!r})")
print(f"Results to   : {RESULTS_JSONL}")
print(f"attn impl    : {ATTN_IMPL!r}")
print(f"static cache : {USE_STATIC_CACHE}")
print(f"sort_by_len  : {SORT_BY_LENGTH}")
print(f"reasoning    : {REASONING}")
print(f"n_votes      : {N_VOTES}")

Model        : google/gemma-4-E2B-it
Benchmarks   : ['arc', 'gsm8k', 'hellaswag', 'truthfulqa', 'winogrande']
Eval combos  : 6 (arc/validation, arc/test, gsm8k/test, hellaswag/validation, truthfulqa/mc_validation, winogrande/validation)
Samples/split: 64
Seed         : 42
Results to   : /content/odiabench_results_google_gemma-4-E2B-it.jsonl


## 5 — Load model (4-bit NF4, multi-GPU aware)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

# ── Faster-kernel flags ──────────────────────────────────────────────────
# bf16 on Ampere+ (compute capability ≥ 8.0 — A100, L4, A10G, H100, …);
# fall back to fp16 on Turing (T4) where bf16 isn't supported in hardware.
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    COMPUTE_DTYPE, _dtype_name = torch.bfloat16, "bfloat16"
else:
    COMPUTE_DTYPE, _dtype_name = torch.float16, "float16"

# TF32 matmuls + cuDNN autotune — free win on Ampere+, no-op on older cards.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True

print(f"Compute dtype: {_dtype_name}  |  TF32: on  |  cuDNN benchmark: on")

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit             = LOAD_IN_4BIT,
    bnb_4bit_quant_type      = "nf4",
    bnb_4bit_compute_dtype   = COMPUTE_DTYPE,
    bnb_4bit_use_double_quant= True,
) if LOAD_IN_4BIT else None

# Try AutoProcessor first (Gemma 4 needs it), fall back to AutoTokenizer
try:
    tokenizer = AutoProcessor.from_pretrained(MODEL_ID)
    print(f"Loaded processor: {type(tokenizer).__name__}")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    print(f"Loaded tokenizer: {type(tokenizer).__name__}")


_model_kwargs = dict(
    quantization_config = bnb_cfg,
    device_map          = "auto",
    torch_dtype         = COMPUTE_DTYPE,
)

_attn_chain = (
    ["flash_attention_2", "sdpa", None]
    if ATTN_IMPL == "auto"
    else [ATTN_IMPL]
)
_attn_used = "default"
model = None
for _impl in _attn_chain:
    try:
        if _impl is None:
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
            _attn_used = "default"
        else:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, attn_implementation=_impl, **_model_kwargs,
            )
            _attn_used = _impl
        break
    except (ValueError, TypeError, ImportError, OSError):
        if ATTN_IMPL != "auto":
            raise
        continue
if model is None:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
    _attn_used = "default"
model.eval()
print(f"Loaded: {MODEL_ID}  |  attn: {_attn_used}")


# Unwrap inner tokenizer if we got a Processor (needed for pad/eos/decode)
_inner_tok = getattr(tokenizer, "tokenizer", tokenizer)

if _inner_tok.pad_token is None:
    _inner_tok.pad_token = _inner_tok.eos_token
_inner_tok.padding_side = "left"

DEVICE = next(model.parameters()).device
print(f"Model device : {DEVICE}")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM used    : {allocated:.2f} GB")


# ── torch.compile model.forward ──────────────────────────────────────────
# Compile only model.forward (model.generate has Python control flow that
# trips up Dynamo).  USE_STATIC_CACHE switches to fullgraph + static KV.
_torch_compile_applied = False
_static_cache_applied = False
if USE_TORCH_COMPILE:
    _torch_ver = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    if _torch_ver < (2, 4):
        print(f"[warn] torch.compile skipped: torch {torch.__version__} < 2.4")
    else:
        _compile_kwargs = dict(
            mode=TORCH_COMPILE_MODE,
            dynamic=True,
            fullgraph=False,
        )
        if USE_STATIC_CACHE:
            try:
                model.generation_config.cache_implementation = "static"
                _compile_kwargs = dict(
                    mode=TORCH_COMPILE_MODE,
                    dynamic=False,
                    fullgraph=True,
                )
                _static_cache_applied = True
                print("static KV-cache: generation_config.cache_implementation='static'")
            except Exception as e:
                print(f"[warn] static KV-cache skipped ({type(e).__name__}: {e}); "
                      "BnB 4-bit + variable batch shapes often break fullgraph compile")
        try:
            model.forward = torch.compile(model.forward, **_compile_kwargs)
            _torch_compile_applied = True
            print(f"torch.compile: applied to model.forward  "
                  f"(mode={TORCH_COMPILE_MODE!r}, "
                  f"dynamic={_compile_kwargs['dynamic']}, "
                  f"fullgraph={_compile_kwargs['fullgraph']})")
            print("  First forward pass will be slow (graph capture); "
                  "subsequent calls run on the compiled graph.")
        except Exception as e:
            if _static_cache_applied:
                print(f"[warn] fullgraph compile failed ({type(e).__name__}: {e}); "
                      "retrying without static cache")
                _static_cache_applied = False
                try:
                    model.generation_config.cache_implementation = None
                except Exception:
                    pass
                try:
                    model.forward = torch.compile(
                        model.forward,
                        mode=TORCH_COMPILE_MODE,
                        dynamic=True,
                        fullgraph=False,
                    )
                    _torch_compile_applied = True
                    print("torch.compile: applied (fallback dynamic=True, fullgraph=False)")
                except Exception as e2:
                    print(f"[warn] torch.compile failed ({type(e2).__name__}: {e2}); "
                          "falling back to eager mode")
            else:
                print(f"[warn] torch.compile failed ({type(e).__name__}: {e}); "
                      "falling back to eager mode")
else:
    print("torch.compile: disabled (USE_TORCH_COMPILE=False)")


Compute dtype: bfloat16  |  TF32: on  |  cuDNN benchmark: on


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Loaded: google/gemma-4-E2B-it  |  attn: sdpa
Model device : cuda:0
VRAM used    : 6.28 GB


## 6 — Define `generate_fn`

In [6]:
import logging
import re

from transformers import ProcessorMixin, StoppingCriteria, StoppingCriteriaList

# ── Silence Inductor cudagraph-partition spam ────────────────────────────
# torch.compile(mode="reduce-overhead") logs "cudagraph partition due to
# non gpu ops" once per partition per generate step.  With sliding-window
# KV caches that's hundreds of lines; informational, not actionable.
logging.getLogger("torch._inductor.cudagraph_trees").setLevel(logging.ERROR)
logging.getLogger("torch._inductor.compile_fx").setLevel(logging.ERROR)

_is_processor = isinstance(tokenizer, ProcessorMixin)


def _apply_chat_template(prompts: list[str]) -> list[str]:
    """Wrap each prompt in the model's chat template."""
    tpl = _inner_tok if hasattr(_inner_tok, "apply_chat_template") else tokenizer
    if not hasattr(tpl, "apply_chat_template"):
        return prompts
    try:
        return [
            tpl.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in prompts
        ]
    except Exception:
        return prompts


def _tokenize(texts: list[str]):
    """Tokenize handling both Processor (Gemma 4) and plain Tokenizer.

    For Processors we filter to input_ids + attention_mask only, because
    the Processor may also return pixel_values=None or other keys that
    would confuse model.generate(**enc).
    """
    if _is_processor:
        raw = tokenizer(text=texts, return_tensors="pt",
                        padding=True, truncation=True, max_length=1024)
        from transformers import BatchEncoding
        return BatchEncoding({
            k: v for k, v in raw.items()
            if k in ("input_ids", "attention_mask") and v is not None
        })
    return _inner_tok(texts, return_tensors="pt",
                      padding=True, truncation=True, max_length=1024)


_BOXED_CLOSED_RE = re.compile(r"\\boxed\s*\{\s*[^{}]*?\}")


class _McqEarlyStopCriteria(StoppingCriteria):
    """Halt when every row closes \\boxed{...} (always) or emits a blank line (plain MCQ only)."""

    _DECODE_TAIL = 64

    def __init__(
        self,
        tok,
        prompt_len: int,
        check_every: int = 4,
        *,
        reasoning: bool = False,
    ) -> None:
        self.tok = tok
        self.prompt_len = prompt_len
        self.check_every = check_every
        self.reasoning = reasoning
        self._step = 0
        self._done: list[bool] | None = None

    def _row_done(self, text: str) -> bool:
        if not self.reasoning and "\n\n" in text:
            return True
        return bool(_BOXED_CLOSED_RE.search(text))

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        self._step += 1
        if self._step % self.check_every != 0:
            return False
        n = input_ids.shape[0]
        if self._done is None:
            self._done = [False] * n
        new_ids = input_ids[:, self.prompt_len:]
        for i in range(n):
            if self._done[i]:
                continue
            tail = new_ids[i, -self._DECODE_TAIL:].tolist()
            if self._row_done(self.tok.decode(tail, skip_special_tokens=True)):
                self._done[i] = True
        return all(self._done)


def make_generate_fn(
    max_new_tokens: int = 512,
    early_stop_on_box: bool = False,
    reasoning: bool = False,
):
    @torch.inference_mode()
    def _generate(prompts: list[str]) -> list[str]:
        templated = _apply_chat_template(prompts)
        enc = _tokenize(templated).to(DEVICE)

        gen_kwargs = dict(
            max_new_tokens  = max_new_tokens,
            do_sample       = False,
            temperature     = 1.0,
            pad_token_id    = _inner_tok.pad_token_id or _inner_tok.eos_token_id,
            eos_token_id    = _inner_tok.eos_token_id,
        )
        if early_stop_on_box:
            prompt_len = enc["input_ids"].shape[1]
            gen_kwargs["stopping_criteria"] = StoppingCriteriaList([
                _McqEarlyStopCriteria(
                    _inner_tok, prompt_len, reasoning=reasoning,
                ),
            ])

        # CUDA-graph step boundary: tells cudagraph_trees that each
        # generate() is a fresh inference step.  Without this, models
        # with in-place KV-cache mutation (Gemma 4 sliding window,
        # Llama 3.x, ...) crash with "accessing tensor output of
        # CUDAGraphs that has been overwritten by a subsequent run".
        # No-op when torch.compile / reduce-overhead is not active.
        if _torch_compile_applied:
            try:
                torch.compiler.cudagraph_mark_step_begin()
            except Exception:
                pass

        out = model.generate(**enc, **gen_kwargs)

        prompt_len = enc["input_ids"].shape[1]
        gen_ids    = out[:, prompt_len:]
        return _inner_tok.batch_decode(gen_ids, skip_special_tokens=True)

    return _generate


generate_mcq   = make_generate_fn(max_new_tokens=32, early_stop_on_box=True, reasoning=REASONING)
generate_gsm8k = make_generate_fn(max_new_tokens=512, early_stop_on_box=False, reasoning=REASONING)

print("generate_fn ready")

test_out = generate_mcq(["ଏହି ବାକ୍ୟ ର ଅର୍ଥ କ'ଣ?"])
print("Sanity check output:", repr(test_out[0][:80]))


generate_fn ready
Sanity check output: 'ଦୟାକରି ଆପଣ ଯଦି **ବାକ୍ୟଟି** ଦେବେ, ତେବେ ମୁଁ ତା'


## 7 — Run evaluation

In [7]:
# Per-benchmark generate function + batch size.
# GSM8K generates up to 512 tokens (chain-of-thought) so it needs a
# smaller batch to avoid OOM; MCQ benchmarks generate ≤32 tokens.
_BENCH_CFG = {
    "gsm8k":      (generate_gsm8k, max(1, BATCH_SIZE // 4)),
    "arc":        (generate_mcq,   BATCH_SIZE),
    "truthfulqa": (generate_mcq,   BATCH_SIZE),
    "winogrande": (generate_mcq,   BATCH_SIZE),
    "hellaswag":  (generate_mcq,   BATCH_SIZE),
}


# ── Hub loader ───────────────────────────────────────────────────────────
# We can't use the library's run_eval(..., hf_repo=...) path because its
# internal _load_from_hub has a hardcoded split_map (one split per
# benchmark), which would prevent us from evaluating arc/validation and
# arc/test independently.
#
# We also can't use datasets.load_dataset() directly: several OdiaBench
# Hub repos have a parquet/declared-schema mismatch on list-typed columns
# (e.g. odia-hellaswag declares `all_endings: string` in the card while
# the parquet actually stores `list<string>`), which makes load_dataset
# fail with `Couldn't cast array of type list<element: string> to string`.
#
# Reading the parquet shards directly via the `hf://` filesystem bypasses
# the bad schema declaration and gives us back the real list-typed columns
# (as numpy ndarrays, which we then normalise to plain lists so the
# scorers' `.index(...)` calls work).
import random
import pandas as pd
from huggingface_hub import list_repo_files

_LIST_FIELDS = ("all_endings", "mc1_choices", "mc1_labels")


def _hub_parquet_urls(repo: str, hf_split: str) -> list[str]:
    """Resolve every parquet shard for <repo>@<hf_split>."""
    files = list_repo_files(repo, repo_type="dataset")
    shards = [f for f in files
              if f.startswith(f"data/{hf_split}-") and f.endswith(".parquet")]
    if not shards:
        raise FileNotFoundError(
            f"No parquet shards matching data/{hf_split}-* in {repo}. "
            f"Available files: {files[:10]}"
        )
    return [f"hf://datasets/{repo}/{s}" for s in shards]


def _row_to_dict(series: pd.Series) -> dict:
    """Series → dict, with numpy-array list fields converted to Python lists
    so scorers that call ``.index(...)`` on them (e.g. TruthfulQA) work."""
    d = series.to_dict()
    for k in _LIST_FIELDS:
        if k in d and not isinstance(d[k], list):
            d[k] = list(d[k])
    return d


def _gen_with_optional_sort(gen_fn, prompts, sort_by_length):
    """Call gen_fn(prompts), optionally length-bucketed then restored."""
    if not sort_by_length or len(prompts) <= 1:
        return gen_fn(prompts)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]), reverse=True)
    sorted_prompts = [prompts[i] for i in order]
    sorted_preds = gen_fn(sorted_prompts)
    if len(sorted_preds) != len(prompts):
        raise ValueError(
            f"gen_fn returned {len(sorted_preds)} outputs for {len(prompts)} prompts"
        )
    preds = [None] * len(prompts)
    for si, oi in enumerate(order):
        preds[oi] = sorted_preds[si]
    return preds


def hub_run_eval(benchmark, hf_repo, hf_split, gen_fn, *,
                 n_samples=None, seed=42, batch_size=64,
                 sort_by_length=False, reasoning=False, n_votes=1,
                 verbose=True):
    """Fetch <hf_repo>@<hf_split> via parquet, score every row via the
    library's build_prompt + score helpers.  Same contract as
    odia_eval.run_eval (incl. _strip_prompt safety net + n_votes
    self-consistency) without the hardcoded-split limitation."""
    if n_votes < 1:
        raise ValueError(f"n_votes must be >= 1 (got {n_votes})")

    urls = _hub_parquet_urls(hf_repo, hf_split)
    dfs = [pd.read_parquet(u) for u in urls]
    df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    rows = [_row_to_dict(df.iloc[i]) for i in range(len(df))]

    if n_samples is not None and n_samples < len(rows):
        rng = random.Random(seed)
        rows = rng.sample(rows, n_samples)

    total = len(rows)
    results = []
    for i in range(0, total, batch_size):
        batch = rows[i:i + batch_size]
        prompts = [build_prompt(benchmark, r, reasoning=reasoning) for r in batch]

        if n_votes == 1:
            preds = _gen_with_optional_sort(gen_fn, prompts, sort_by_length)
            if len(preds) != len(prompts):
                raise ValueError(
                    f"gen_fn returned {len(preds)} outputs for {len(prompts)} prompts"
                )
            for r, p, pred in zip(batch, prompts, preds):
                # Mirror run_eval: strip the prompt prefix so generate_fns
                # that return prompt+completion (e.g. vLLM without slicing)
                # don't bias the regex extractors.
                completion = _strip_prompt(pred, p)
                sr = score(benchmark, completion, r)
                results.append(EvalRow(
                    benchmark=benchmark, row_id=r["id"], prompt=p, score_result=sr,
                ))
        else:
            # Replicate each prompt n_votes times so a single gen_fn call
            # covers the whole batch (preserves any kv-cache batching).
            expanded = [p for p in prompts for _ in range(n_votes)]
            all_preds = _gen_with_optional_sort(gen_fn, expanded, sort_by_length)
            if len(all_preds) != len(expanded):
                raise ValueError(
                    f"gen_fn returned {len(all_preds)} outputs for {len(expanded)} "
                    f"expanded prompts (batch={len(prompts)} x n_votes={n_votes})"
                )
            for j, (r, p) in enumerate(zip(batch, prompts)):
                raw_chunk = all_preds[j * n_votes : (j + 1) * n_votes]
                completions = [_strip_prompt(pp, p) for pp in raw_chunk]
                sr = _aggregate_majority_vote(benchmark, completions, r)
                results.append(EvalRow(
                    benchmark=benchmark, row_id=r["id"], prompt=p, score_result=sr,
                ))

        if verbose:
            correct = sum(rr.correct for rr in results)
            done = min(i + batch_size, total)
            vote_tag = f" (n_votes={n_votes})" if n_votes > 1 else ""
            print(f"[{benchmark}/{hf_split}{vote_tag}] {done}/{total}  "
                  f"acc: {correct}/{len(results)} ({100 * correct / len(results):.1f}%)")
    return results


# ── Run every (benchmark, split) combo ───────────────────────────────────
# Skipped splits are collected here so a single bad split (network blip,
# schema mismatch, OOM) doesn't disappear into print noise — they're
# printed at the end and recorded in the .meta.json sidecar.
all_results: dict[str, list] = {}
skipped_splits: list[dict] = []

for benchmark, split_name, repo, hf_split in EVAL_COMBOS:
    gen_fn, bs = _BENCH_CFG[benchmark]
    key = f"{benchmark}/{split_name}"
    print(f"\n{'='*60}")
    print(f" {key.upper():<40s} | n={N_SAMPLES} | bs={bs} | n_votes={N_VOTES}")
    print(f" hub: {repo} @ {hf_split}")
    print(f"{'='*60}")

    try:
        rows = hub_run_eval(
            benchmark, repo, hf_split, gen_fn,
            n_samples=N_SAMPLES, seed=SEED, batch_size=bs,
            sort_by_length=SORT_BY_LENGTH, reasoning=REASONING,
            n_votes=N_VOTES, verbose=True,
        )
    except Exception as e:
        print(f"[skip] {key}: {type(e).__name__}: {e}")
        skipped_splits.append({
            "key": key, "repo": repo, "hf_split": hf_split,
            "error_type": type(e).__name__, "error": str(e),
        })
        continue

    for r in rows:
        r.benchmark = key
    all_results[key] = rows

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if skipped_splits:
    print(f"\n{'!'*60}")
    print(f" {len(skipped_splits)} SPLIT(S) SKIPPED — results report is incomplete:")
    for s in skipped_splits:
        print(f"   - {s['key']}: {s['error_type']}: {s['error']}")
    print(f"{'!'*60}")


 ARC/VALIDATION                           | n=64 | bs=64
 hub: tripathysagar/odia-arc @ validation
[arc/validation] 64/64  acc: 29/64 (45.3%)

 ARC/TEST                                 | n=64 | bs=64
 hub: tripathysagar/odia-arc @ test
[arc/test] 64/64  acc: 32/64 (50.0%)

 GSM8K/TEST                               | n=64 | bs=16
 hub: tripathysagar/odia-gsm8k @ test
[gsm8k/test] 16/64  acc: 6/16 (37.5%)
[gsm8k/test] 32/64  acc: 9/32 (28.1%)
[gsm8k/test] 48/64  acc: 13/48 (27.1%)
[gsm8k/test] 64/64  acc: 20/64 (31.2%)

 HELLASWAG/VALIDATION                     | n=64 | bs=64
 hub: tripathysagar/odia-hellaswag @ validation
[hellaswag/validation] 64/64  acc: 29/64 (45.3%)

 TRUTHFULQA/MC_VALIDATION                 | n=64 | bs=64
 hub: tripathysagar/odia-truthfulqa-mc @ validation
[truthfulqa/validation] 64/64  acc: 23/64 (35.9%)

 WINOGRANDE/VALIDATION                    | n=64 | bs=64
 hub: tripathysagar/odia-winogrande @ validation
[winogrande/validation] 64/64  acc: 44/64 (68.8%)


## 8 — Results

In [8]:
from odia_eval import accuracy_with_ci, chance_level


def split_report(results_by_key: dict, alpha: float = 0.05) -> str:
    """Multi-line accuracy table that understands composite '<benchmark>/<split>'
    keys.  Chance level is looked up via the leading benchmark name, so e.g.
    'arc/test' and 'arc/validation' both report chance 25.1%.
    """
    title = f"OdiaBench Evaluation Results  ({int((1 - alpha) * 100)}% Wilson CI)"
    lines = [title, "=" * 72]
    accs: list[float] = []
    chances: list[float] = []
    for key in sorted(results_by_key):
        rows = results_by_key[key]
        if not rows:
            continue
        base = key.split("/", 1)[0]
        acc, lo, hi, correct, total = accuracy_with_ci(rows, alpha=alpha)
        chance = chance_level(base)
        delta = (acc - chance) * 100
        accs.append(acc * 100)
        chances.append(chance * 100)
        lines.append(
            f"{key:<24s} {correct:>5}/{total:<5}  "
            f"{acc * 100:5.1f}%  [{lo * 100:5.1f}–{hi * 100:5.1f}]  "
            f"chance {chance * 100:4.1f}%  Δ {delta:+5.1f}"
        )
    if accs:
        mean_acc = sum(accs) / len(accs)
        mean_chance = sum(chances) / len(chances)
        lines.append("-" * 72)
        lines.append(
            f"{'MEAN':<24s} {'':>11s}  {mean_acc:5.1f}%  "
            f"{'':<14s}  {'':<13s}  Δ {mean_acc - mean_chance:+5.1f}"
        )
    return "\n".join(lines)


print(f"\nModel: {MODEL_ID}\n")
print(split_report(all_results))


Model: google/gemma-4-E2B-it

OdiaBench Evaluation Results  (95% Wilson CI)
arc/test                    32/64      50.0%  [ 38.1– 61.9]  chance 25.1%  Δ +24.9
arc/validation              29/64      45.3%  [ 33.7– 57.4]  chance 25.1%  Δ +20.2
gsm8k/test                  20/64      31.2%  [ 21.2– 43.4]  chance  0.0%  Δ +31.2
hellaswag/validation        29/64      45.3%  [ 33.7– 57.4]  chance 25.0%  Δ +20.3
truthfulqa/mc_validation    23/64      35.9%  [ 25.3– 48.2]  chance 22.6%  Δ +13.3
winogrande/validation       44/64      68.8%  [ 56.6– 78.8]  chance 50.0%  Δ +18.8
------------------------------------------------------------------------
MEAN                                   46.1%                                 Δ +21.5


## 9 — Save results

In [9]:
import json
from datetime import datetime, timezone
from odia_eval import save_results, load_results

# Per-row results (prompt + gold + extracted + prediction + correct)
n_rows = save_results(all_results, RESULTS_JSONL)
print(f"Saved {n_rows} row-level records → {RESULTS_JSONL}")

# Companion metadata sidecar (one row, easy to grep across runs)
meta = {
    "model_id":   MODEL_ID,
    "n_samples":  N_SAMPLES,
    "seed":       SEED,
    "batch_size": BATCH_SIZE,
    "load_4bit":  LOAD_IN_4BIT,
    "torch_compile":         bool(_torch_compile_applied),
    "torch_compile_mode":    TORCH_COMPILE_MODE if _torch_compile_applied else None,
    "attn_implementation":   _attn_used,
    "use_static_cache":      bool(_static_cache_applied),
    "reasoning":             REASONING,
    "n_votes":               N_VOTES,
    "early_stop_on_box":     dict(_EARLY_STOP_ON_BOX),
    "sort_by_length":        SORT_BY_LENGTH,
    "odia_eval_version":     odia_eval.__version__,
    "gpu":        HW["gpu_name"],
    "n_gpus":     HW["n_gpus"],
    "timestamp":  datetime.now(timezone.utc).isoformat(),
    "evaluated_splits": sorted(all_results.keys()),
    "skipped_splits":   skipped_splits,
}
meta_path = str(RESULTS_JSONL).rsplit(".", 1)[0] + ".meta.json"
with open(meta_path, "w", encoding="utf-8") as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)
print(f"Saved run metadata    → {meta_path}")

# Sanity: round-trip the results and confirm the report is identical.
# Use the composite-key-aware split_report defined above so the assert
# covers every (benchmark, split) entry, not just the leading benchmark.
reloaded = load_results(RESULTS_JSONL)
assert split_report(all_results) == split_report(reloaded), "round-trip mismatch"
print("Round-trip OK — re-score later with load_results() without re-running generation.")

Saved 384 row-level records → /content/odiabench_results_google_gemma-4-E2B-it.jsonl
Saved run metadata    → /content/odiabench_results_google_gemma-4-E2B-it.meta.json
Round-trip OK — re-score later with load_results() without re-running generation.


## 10 — Per-benchmark breakdown (optional deeper look)

In [10]:
# Show a few wrong predictions to spot failure modes
for bm, rows in all_results.items():
    wrong = [r for r in rows if not r.correct][:3]
    if not wrong:
        continue
    print(f"\n── {bm} — first 3 wrong ──")
    for r in wrong:
        print(f"  id={r.row_id}  extracted={r.score_result.extracted!r}  "
              f"gold={r.score_result.gold!r}")
        print(f"  prediction: {r.score_result.prediction[:120]!r}")


── arc/validation — first 3 wrong ──
  id=57  extracted='D'  gold='B'
  prediction: 'ସଠିକ ଉତ୍ତର: **D: କେନ୍ଦ୍ରକ**\n\n**ବ୍ୟ ವಿವರಣೆ:**\n\n* **ପ୍ରାଣ'
  id=12  extracted='D'  gold='A'
  prediction: 'ସଠିକ ଉତ୍ତର ହେଉଛି **D: ଖାଦ୍ୟ ସମାନ ତାପମାତ୍ରାରେ'
  id=140  extracted='C'  gold='A'
  prediction: 'ସଠିକ ଉତ୍ତର: **C: ଧନାତ୍ମକ ଆଧାରିତ**\n\n**ବ୍ୟ ವಿವರಣೆ:**\n\n'

── arc/test — first 3 wrong ──
  id=228  extracted='A'  gold='B'
  prediction: 'ସଠିକ ଉତ୍ତର: **A: ଏକ୍ସୋଥର୍ମିକ୍।**\n\n**ବ୍ୟ ವಿವರಣೆ:**\n\n*'
  id=501  extracted='C'  gold='A'
  prediction: 'ସଠିକ ଉତ୍ତର: **C**\n\n**ବ୍ୟାଖ୍ୟା:**\n\n* **ପ୍ରତ୍ୟୁହ୍ୟପ'
  id=191  extracted='C'  gold='B'
  prediction: 'ସଠିକ ଉତ୍ତର ହେଉଛି **C: ଅପରାହ୍ନ 4:05**।\n\n**ବ୍ୟ'

── gsm8k/test — first 3 wrong ──
  id=1309  extracted='1450'  gold='2280'
  prediction: "ଆମେ ପ୍ରଥମେ ଆଲେକ୍ସାଣ୍ଡ୍ରାଠାର ରୋଜଗାର୍ ଖୋଜିବା:\nଆଲେକ୍ସାଣ୍ଡ୍ରାଠାର ରୋଜଗାର୍ = $320 + $430 = $750\n\nତା' ପରେ, ମରିୟମ୍ ସାରାହଠାର ରୋଜଗ"
  id=228  extracted='360'  gold='1'
  prediction: 'ଏହା ଏକ ସମୟ ଓ କାର୍ଯ୍ୟ ସମ୍ବନ୍ଧୀୟ ଗଣିତ ପ୍ରଶ୍ନ। ଏହ